# KNN Regression

**Goal:** Predict genome-wide perturbation effects (logFC) for TFs not in Perturb-seq, using GenePT embeddings + kNN regression.

**Logic:**
- 409 TFs in Perturb-seq (Rest) have *known* logFC vectors → training set
- Unseen TFs (pySCENIC candidates not in Perturb-seq) get GenePT embedding → find k nearest neighbors among known TFs → average their logFC vectors → predicted effect

**Steps:**
1. Download & load GenePT embeddings
2. Check coverage for our TFs
3. Build embedding matrix (X) for known Perturb-seq TFs
4. Build response matrix (Y) from Perturb-seq logFC data
5. 5-fold cross-validation to pick best k
6. Predict for unseen TFs
7. Filter to edges and cross-reference with pySCENIC/scPRINT2


Obtaining the Gene PT embeddings

In [3]:
import os

if not os.path.exists('GenePT_gene_embedding_ada_text.pickle'):
    !wget -q "https://zenodo.org/records/10833191/files/GenePT_emebdding_v2.zip"
    !unzip -q GenePT_emebdding_v2.zip
    print("Downloaded and extracted.")
else:
    print("Already downloaded.")

Downloaded and extracted.


In [14]:
import pickle
import numpy as np

with open('/content/GenePT_emebdding_v2/GenePT_gene_embedding_ada_text.pickle', 'rb') as f:
    genept = pickle.load(f)

example_gene = list(genept.keys())[0]
example_vec  = genept[example_gene]

# GenePT stores embeddings as plain Python lists, not numpy arrays
print(f"Total genes in GenePT:  {len(genept)}")
print(f"Example key:            {example_gene}")
print(f"Value type:             {type(example_vec)}")
print(f"Embedding length:       {len(example_vec)}")

Total genes in GenePT:  93800
Example key:            A1BG
Value type:             <class 'list'>
Embedding length:       1536


Check GenePT coverage for our TFs

In [23]:
import pandas as pd

# Lambert full TF list
lambert = pd.read_csv('/content/drive/MyDrive/benchmarking_paper_v2/reference/humanTFs_1639_clean.csv')
lambert_tfs = lambert['gene_symbol'].dropna().str.upper().tolist()

# Perturb-seq known TFs (Rest condition — our training set)
perturb_rest = pd.read_csv('/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs/perturb_net_tfs_only_Rest_filtered.csv')
perturb_tfs = perturb_rest['source_TF'].str.upper().unique().tolist()

# Coverage check
lambert_in_genept   = [tf for tf in lambert_tfs if tf in genept]
perturb_in_genept   = [tf for tf in perturb_tfs if tf in genept]
missing_from_genept = [tf for tf in lambert_tfs if tf not in genept]

print(f"Lambert TFs total:               {len(lambert_tfs)}")
print(f"Lambert TFs WITH GenePT emb:     {len(lambert_in_genept)}")
print(f"Perturb-seq TFs WITH GenePT emb: {len(perturb_in_genept)}")
print(f"Lambert TFs MISSING from GenePT: {len(missing_from_genept)}")
print(f"\nMissing examples: {missing_from_genept[:10]}")

Lambert TFs total:               1639
Lambert TFs WITH GenePT emb:     1623
Perturb-seq TFs WITH GenePT emb: 409
Lambert TFs MISSING from GenePT: 16

Missing examples: ['AC008770.3', 'AC023509.3', 'AC092835.1', 'AC138696.1', 'ARNTL', 'BORCS8-MEF2B', 'C11ORF95', 'CENPBD1', 'CTCFL', 'DUX1']


Build embedding matrix X (training embeddings)

In [24]:
import numpy as np

# Use only Perturb-seq TFs that have a GenePT embedding (all 409 do)
train_tfs = [tf for tf in perturb_tfs if tf in genept]

# Stack into matrix: shape (n_tfs, 1536)
X = np.array([genept[tf] for tf in train_tfs])

print(f"Training TFs:      {len(train_tfs)}")
print(f"X shape:           {X.shape}")   # expect (409, 1536)
print(f"X dtype:           {X.dtype}")

Training TFs:      409
X shape:           (409, 1536)
X dtype:           float64


Building response matrix - Y

In [29]:
# Load Perturb-seq edges for Rest condition
perturb_rest = pd.read_csv('/content/drive/MyDrive/benchmarking_paper/datasets/perturbnet_outputs/perturb_net_tfs_only_Rest_filtered.csv')

# Uppercase TF column to match train_tfs
perturb_rest['source_TF'] = perturb_rest['source_TF'].str.upper()

# Keep only TFs that are in our training set (have GenePT embeddings)
perturb_rest = perturb_rest[perturb_rest['source_TF'].isin(train_tfs)]

# Pivot: rows = TFs, columns = target genes, values = logFC
# TFs with no edge to a gene get 0 (no change)
Y = perturb_rest.pivot_table(index='source_TF', columns='target_gene', values='log2FC', aggfunc='mean', fill_value=0.0)

# Reorder rows to match train_tfs order (so X[i] and Y[i] are the same TF)
Y = Y.reindex(train_tfs).fillna(0.0)

print(f"Y shape:        {Y.shape}")   # expect (409, ~8000+ genes)
print(f"Y TFs (rows):   {Y.index.tolist()[:5]}")
print(f"Y genes (cols): {Y.columns.tolist()[:5]}")

Y shape:        (409, 8288)
Y TFs (rows):   ['ADNP', 'ADNP2', 'AEBP2', 'AHDC1', 'AHR']
Y genes (cols): ['AAAS', 'AACS', 'AAGAB', 'AAK1', 'AAMP']


## Step 6 — 5-fold cross-validation to pick best k

Sweep k ∈ {3, 5, 10, 15, 25}. Metric: median Pearson r per TF across folds.
Also compute baselines: Train Mean, No Change.

In [30]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import KFold
from scipy.stats import pearsonr
import pandas as pd
import numpy as np

# L2-normalize X so euclidean distance ≈ cosine similarity
X_norm = X / np.linalg.norm(X, axis=1, keepdims=True)
Y_vals = Y.values  # numpy array (409, n_genes)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
k_values = [3, 5, 10, 15, 25]
results = []

for k in k_values:
    fold_pearsons = []
    for fold, (tr_idx, te_idx) in enumerate(kf.split(X_norm)):
        model = KNeighborsRegressor(n_neighbors=k, weights='distance', metric='euclidean')
        model.fit(X_norm[tr_idx], Y_vals[tr_idx])
        Y_pred = model.predict(X_norm[te_idx])

        for i, row_idx in enumerate(te_idx):
            y_true = Y_vals[row_idx]
            y_pred = Y_pred[i]
            if y_true.std() > 0 and y_pred.std() > 0:
                r, _ = pearsonr(y_true, y_pred)
                fold_pearsons.append(r)
            else:
                fold_pearsons.append(np.nan)

    results.append({'k': k, 'median_pearson': np.nanmedian(fold_pearsons),
                    'mean_pearson': np.nanmean(fold_pearsons)})
    print(f"k={k:2d}  median Pearson: {np.nanmedian(fold_pearsons):.4f}")

# Baseline: predict the training mean for every test TF
baseline_pearsons = []
for tr_idx, te_idx in kf.split(X_norm):
    train_mean = Y_vals[tr_idx].mean(axis=0)
    for row_idx in te_idx:
        y_true = Y_vals[row_idx]
        if y_true.std() > 0 and train_mean.std() > 0:
            r, _ = pearsonr(y_true, train_mean)
            baseline_pearsons.append(r)
        else:
            baseline_pearsons.append(np.nan)

print(f"\nBaseline (Train Mean) median Pearson: {np.nanmedian(baseline_pearsons):.4f}")

cv_df = pd.DataFrame(results)
print(f"\n{cv_df}")

k= 3  median Pearson: 0.0000
k= 5  median Pearson: 0.0000
k=10  median Pearson: 0.0003
k=15  median Pearson: 0.0007
k=25  median Pearson: 0.0011

Baseline (Train Mean) median Pearson: 0.0126

    k  median_pearson  mean_pearson
0   3        0.000036      0.005143
1   5        0.000035      0.008304
2  10        0.000278      0.010532
3  15        0.000671      0.011760
4  25        0.001055      0.013190


## Predict for unseen TFs with k = 25

In [37]:
import os
os.makedirs('/content/drive/MyDrive/benchmarking_paper/datasets/knn_genept_outputs', exist_ok=True)

pred_df = pd.DataFrame(Y_pred, index=unseen_tfs, columns=Y.columns)
pred_df.index.name = 'TF'
pred_df.to_csv('/content/drive/MyDrive/benchmarking_paper/datasets/knn_genept_outputs/knn_predicted_logFC_wide_Rest.csv')
print("Saved wide matrix.")

edges = pred_df.stack().reset_index()
edges.columns = ['TF', 'target_gene', 'predicted_logFC']
edges = edges[edges['predicted_logFC'].abs() > 0.5]
edges['condition'] = 'Rest'
edges = edges.sort_values('predicted_logFC', key=abs, ascending=False)

print(f"Edges with |logFC| > 0.5: {len(edges)}")
print(edges.head(10))

Saved wide matrix.
Edges with |logFC| > 0.5: 27
          TF target_gene  predicted_logFC condition
185244  EGR1        GZMK         4.214790      Rest
185902  EGR1       KLRG1         2.813733      Rest
2753    ATF2        GNG8         2.533599      Rest
183404  EGR1        CCL5         1.869446      Rest
182678  EGR1        AOAH         1.671183      Rest
790276  NRF1        H1-5        -1.366097      Rest
790285  NRF1       H2BC9        -1.204676      Rest
790290  NRF1        H4C3        -1.051204      Rest
790273  NRF1        H1-2        -0.923076      Rest
793248  NRF1        RRM2        -0.831938      Rest


In [38]:
edges.to_csv('/content/drive/MyDrive/benchmarking_paper/datasets/knn_genept_outputs/knn_predicted_edges_Rest.csv', index=False)
print(f"Saved {len(edges)} edges for {edges['TF'].nunique()} TFs.")
print(f"\nTFs with predicted edges: {sorted(edges['TF'].unique().tolist())}")

Saved 27 edges for 9 TFs.

TFs with predicted edges: ['ATF2', 'EGR1', 'ELK1', 'NR1H3', 'NRF1', 'PITX1', 'PPARG', 'RFXAP', 'ZNF239']


In [45]:
# Define unseen TFs: pySCENIC TFs NOT in Perturb-seq that have a GenePT embedding
pyscenic = pd.read_csv('/content/drive/MyDrive/benchmarking_paper/datasets/pyscenic/pyscenic_net_edges.csv')
pyscenic_tfs = pyscenic['tf'].str.upper().unique().tolist()

perturb_tfs_set = set(train_tfs)
unseen_tfs = [tf for tf in pyscenic_tfs if tf not in perturb_tfs_set and tf in genept]

print(f"pySCENIC TFs total:          {len(pyscenic_tfs)}")
print(f"pySCENIC TFs in Perturb-seq: {len([tf for tf in pyscenic_tfs if tf.upper() in perturb_tfs_set])}")
print(f"Unseen TFs to predict:       {len(unseen_tfs)}")

# Refit kNN on ALL 409 training TFs (no held-out — use everything for prediction)
best_k = 25
model_final = KNeighborsRegressor(n_neighbors=best_k, weights='distance',metric='euclidean')
model_final.fit(X_norm, Y_vals)

# Build embedding matrix for unseen TFs and predict
X_unseen      = np.array([genept[tf] for tf in unseen_tfs])
X_unseen_norm = X_unseen / np.linalg.norm(X_unseen, axis=1, keepdims=True)
Y_pred        = model_final.predict(X_unseen_norm)  # shape: (149, n_genes)

print(f"\nPrediction matrix shape: {Y_pred.shape}")

# Save wide matrix (TF x gene full logFC predictions)
import os
os.makedirs('/content/drive/MyDrive/benchmarking_paper/datasets/knn_genept_outputs', exist_ok=True)

pred_df = pd.DataFrame(Y_pred, index=unseen_tfs, columns=Y.columns)
pred_df.index.name = 'TF'
pred_df.to_csv('/content/drive/MyDrive/benchmarking_paper/datasets/knn_genept_outputs/knn_predicted_logFC_wide_Rest.csv')
print("Saved wide matrix.")

pySCENIC TFs total:          235
pySCENIC TFs in Perturb-seq: 85
Unseen TFs to predict:       149

Prediction matrix shape: (149, 8288)
Saved wide matrix.


In [46]:
edges = edges[edges['predicted_logFC'].abs() > 0.5]
edges['condition'] = 'Rest'
edges = edges.sort_values('predicted_logFC', key=abs, ascending=False)

print(f"Edges with |logFC| > 0.5: {len(edges)}")
print(f"TFs with predicted edges: {sorted(edges['TF'].unique().tolist())}")
print(edges.head(10))

edges.to_csv('/content/drive/MyDrive/benchmarking_paper/datasets/knn_genept_outputs/knn_predicted_edges_Rest.csv', index=False)
print("\nSaved edges.")

Edges with |logFC| > 0.5: 27
TFs with predicted edges: ['ATF2', 'EGR1', 'ELK1', 'NR1H3', 'NRF1', 'PITX1', 'PPARG', 'RFXAP', 'ZNF239']
          TF target_gene  predicted_logFC condition
185244  EGR1        GZMK         4.214790      Rest
185902  EGR1       KLRG1         2.813733      Rest
2753    ATF2        GNG8         2.533599      Rest
183404  EGR1        CCL5         1.869446      Rest
182678  EGR1        AOAH         1.671183      Rest
790276  NRF1        H1-5        -1.366097      Rest
790285  NRF1       H2BC9        -1.204676      Rest
790290  NRF1        H4C3        -1.051204      Rest
790273  NRF1        H1-2        -0.923076      Rest
793248  NRF1        RRM2        -0.831938      Rest

Saved edges.
